<a href="https://colab.research.google.com/github/Bovi-analytics/Ansci4040FinalClassProject/blob/main/RenameCalvingVideos.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Rename the calving videos to one PascalCase convention

Gives every calving video one name pattern that keeps the **date**, the **start time** (only when it is not midnight), the **pen** and the **part** number:

| Now | New name |
|---|---|
| `2026-11-11 00-00-00 Calving Pen 3 (2).avi` | `CalvingPen3Nov11Part2.avi` |
| `2026-11-11 00-00-00 Calving Pen 3.avi` | `CalvingPen3Nov11Part1.avi` |
| `2026-10-29 09-48-06 Calving Pen 2 (3).avi` | `CalvingPen2Oct29Time094806Part3.avi` |
| `2026-09-26-CalvingPen3(1)_ROI_01-42-00_to_01-57-00_2FPS.avi` | `CalvingPen3Sep26Part1Roi014200To015700Fps2.avi` |
| `2026-09-26-Clip1Pen2(4:37hrs).avi` | `CalvingPen2Sep26Part1Length4h37m.avi` |
| `Copy of 2026-09-26 00-00-00 Calving Pen 3.avi` | `CalvingPen3Sep26Part1Copy.avi` |

`Brackets = True` keeps the numbers in brackets instead: `CalvingPen(3)Nov11Part(2).avi`. The cow is **not** in the file name (a recording can show several cows); cows are labelled per pen in each video's `Identities.csv`: `Pen1Cow1`, `Pen1Cow2`, `Pen2Cow1`, ... (step 06).

**Safe:** only **video files** in `CalvingPenVideos ` and `CalvingPenPipeline` (and their subfolders) are touched; notebooks, spreadsheets and pipeline outputs keep their names. Cell 3 first **only shows** the plan; nothing changes until `Apply = True`. Files whose name is not understood (`Check`), broken shortcuts (`Unreadable`) and clashes (`Conflict`) are left alone. Every rename is written to `CalvingPenPipelineOutputs/RenameLog.csv`, and `Undo = True` puts the old names back. The pipeline still finds a video by its **old** name through that log.

**One person runs this, once, when nobody is running the pipeline** (a video renamed while a step reads it would fail). No GPU needed. Edit cell 2 and rerun; don't add cells. This replaces the earlier `NamingConvention.ipynb`, which pointed to `MyDrive`, renamed every file in the group folder and merged pen and part numbers (`CalvingPen32`).

In [ ]:
# 1. Setup: mount Drive, get the pipeline code from GitHub.
from google.colab import drive
drive.mount("/content/drive")

import os, sys
RepoUrl = "https://github.com/Bovi-analytics/Ansci4040FinalClassProject.git"
RepoBranch = "main"
RepoDir = "/content/Ansci4040FinalClassProject"

if os.path.isdir(RepoDir):
    !git -C "{RepoDir}" fetch -q origin "{RepoBranch}" && git -C "{RepoDir}" checkout -q "{RepoBranch}" && git -C "{RepoDir}" pull -q --ff-only
else:
    !git clone -q --branch "{RepoBranch}" "{RepoUrl}" "{RepoDir}"
sys.path.insert(0, f"{RepoDir}/CalvingPen")

# Only needed after an update: Runtime > Restart session, then rerun from here.
from CalvingPipeline.Rename import PlanRenames, ApplyRenames, UndoRenames, RenameLog
import pandas as pd
pd.set_option("display.max_rows", 500, "display.max_colwidth", 120)

In [ ]:
# 2. Settings: the only cell to edit.
Folders = ["CalvingPenVideos ", "CalvingPenPipeline"]   # inside ANSC4040GroupProject (note the space after Videos)
Brackets = False   # True = CalvingPen(3)Nov11Part(2).avi
Apply = False      # False = only show the plan; True = rename the files marked "Rename"
Undo = False       # True = put the old names back (from RenameLog.csv) instead

In [ ]:
# 3. Run: show the plan; rename only with Apply = True (or undo with Undo = True).
if Undo:
    UndoRenames(RenameLog)
else:
    Plan = PlanRenames(Folders, Brackets=Brackets)
    print(Plan.Status.value_counts().to_string(), "\n")
    display(Plan)
    if Apply:
        Done = ApplyRenames(Plan, RenameLog)
        display(Done)
    else:
        print("PREVIEW ONLY: nothing renamed. Check the NewName column, then set Apply = True and rerun cells 2-3.")

## About this notebook

**Rule** (`CalvingName` in `CalvingPen/CalvingPipeline/Rename.py`): `CalvingPen<pen><Mon><DD>[Time<HHMMSS>]Part<n>[<rest in PascalCase>]` (no year; day with two digits, e.g. Nov01) (pure PascalCase: every number has a word in front, so numbers never run together). The part is the number in brackets after the pen (`Pen 3 (2)`) or after `Clip` (`Clip1Pen2`); without one it is `Part1`. Anything else in the old name (ROI, time window, fps) is kept at the end, so no information is lost.

**Status column:** `Rename` (will be renamed), `Same` (already right), `Check` (no date or pen in the name: left alone, rename by hand), `Unreadable` (broken Drive shortcut or still uploading: left alone), `Conflict` (the new name is already taken or two files would get it: left alone).

**Afterwards:** use the new names in `PipelineSteps` (`VideoFile`, and `VideoStem` = the new name without `.avi`). Old names still work for finding the video. Existing output folders keep their old names; only videos run from now on get new-style folder names.